In [11]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\karth\AppData\Local\Temp\ipykernel_10804\1975300343.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader


In [12]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: 01_cheque_bounce_legal_notice_sample.pdf
  ✓ Loaded 1 pages

Processing: 02_rent_arrears_legal_notice_sample.pdf
  ✓ Loaded 1 pages

Processing: 03_consumer_product_defect_legal_notice_sample.pdf
  ✓ Loaded 1 pages

Processing: 04_property_payment_demand_notice_sample.pdf
  ✓ Loaded 1 pages

Total documents loaded: 4


In [13]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-14T03:58:18+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-14T03:58:18+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\01_cheque_bounce_legal_notice_sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '01_cheque_bounce_legal_notice_sample.pdf', 'file_type': 'pdf'}, page_content='SAMPLE / TEST DOCUMENT — NOT A REAL LEGAL NOTICE\n LEGAL NOTICE UNDER SECTION 138 OF THE NEGOTIABLE\n INSTRUMENTS ACT, 1881\nNotice No.\nLL-TEST-2026-001\nDate\n14 August 2026\nTo\nMr. Rohan Mehta\nAddress\n24, Lake View Road, Bengaluru, Karnataka – 560038\nFrom\nMs. Ananya Rao\nAddress\n17, Green Park Extension, Bengaluru, Karnataka – 560034\nSubject\nDemand for payment of dishonoured cheque\nSir/Madam,\nUnder instructions from and on behalf of the sender, this notice is issued regard

In [14]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [15]:

chunks=split_documents(all_pdf_documents)
chunks

Split 4 documents into 8 chunks

Example chunk:
Content: SAMPLE / TEST DOCUMENT — NOT A REAL LEGAL NOTICE
 LEGAL NOTICE UNDER SECTION 138 OF THE NEGOTIABLE
 INSTRUMENTS ACT, 1881
Notice No.
LL-TEST-2026-001
Date
14 August 2026
To
Mr. Rohan Mehta
Address
24,...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-14T03:58:18+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-14T03:58:18+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\01_cheque_bounce_legal_notice_sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '01_cheque_bounce_legal_notice_sample.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-14T03:58:18+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-14T03:58:18+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\01_cheque_bounce_legal_notice_sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '01_cheque_bounce_legal_notice_sample.pdf', 'file_type': 'pdf'}, page_content='SAMPLE / TEST DOCUMENT — NOT A REAL LEGAL NOTICE\n LEGAL NOTICE UNDER SECTION 138 OF THE NEGOTIABLE\n INSTRUMENTS ACT, 1881\nNotice No.\nLL-TEST-2026-001\nDate\n14 August 2026\nTo\nMr. Rohan Mehta\nAddress\n24, Lake View Road, Bengaluru, Karnataka – 560038\nFrom\nMs. Ananya Rao\nAddress\n17, Green Park Extension, Bengaluru, Karnataka – 560034\nSubject\nDemand for payment of dishonoured cheque\nSir/Madam,\nUnder instructions from and on behalf of the sender, this notice is issued regard

Embeddings and vectorDB

In [16]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3792.72it/s]


Model loaded successfully. Embedding dimension: 384


In [18]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 16


In [19]:
chunks


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-14T03:58:18+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-14T03:58:18+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\01_cheque_bounce_legal_notice_sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '01_cheque_bounce_legal_notice_sample.pdf', 'file_type': 'pdf'}, page_content='SAMPLE / TEST DOCUMENT — NOT A REAL LEGAL NOTICE\n LEGAL NOTICE UNDER SECTION 138 OF THE NEGOTIABLE\n INSTRUMENTS ACT, 1881\nNotice No.\nLL-TEST-2026-001\nDate\n14 August 2026\nTo\nMr. Rohan Mehta\nAddress\n24, Lake View Road, Bengaluru, Karnataka – 560038\nFrom\nMs. Ananya Rao\nAddress\n17, Green Park Extension, Bengaluru, Karnataka – 560034\nSubject\nDemand for payment of dishonoured cheque\nSir/Madam,\nUnder instructions from and on behalf of the sender, this notice is issued regard

In [20]:
### Convert the text to embeddings
####chunks = complete Document objects
##texts = only the actual text extracted from those objects

texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 8 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.54it/s]

Generated embeddings with shape: (8, 384)
Adding 8 documents to vector store...
Successfully added 8 documents to vector store
Total documents in collection: 24


RAG Retriever Pipeline


In [21]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [22]:
rag_retriever

In [23]:
rag_retriever.retrieve("explain RENT ARREARS?")

Retrieving documents for query: 'explain RENT ARREARS?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 55.90it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_d40e5e5f_2',
  'content': 'SAMPLE / TEST DOCUMENT — NOT A REAL LEGAL NOTICE\n SAMPLE LEGAL NOTICE — DEMAND FOR RENT ARREARS\nNotice No.\nLL-TEST-2026-002\nDate\n14 August 2026\nTo\nMr. Arjun Kumar\nAddress\nFlat 302, Sunrise Apartments, Mysuru, Karnataka – 570016\nFrom\nMrs. Kavya Sharma\nAddress\n12, Palace Road, Mysuru, Karnataka – 570001\nSubject\nDemand for alleged unpaid residential rent\nBackground\nThe sender alleges that the recipient occupied a fictional residential premises under a fictional rental\narrangement commencing 1 January 2026.\nAlleged Default\nThe sender alleges that rent of INR 25,000 per month for June and July 2026 remains unpaid, resulting\nin an alleged outstanding principal of INR 50,000.\nDemand\nThe recipient is requested to clear the alleged arrears and communicate with the sender regarding any\ngenuine dispute concerning the calculation or tenancy terms.\nConsequences\nThe sender states that appropriate remedies may be pursued if the alleged

RAG Pipeline- VectorDB To LLM Output Generation

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [25]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [26]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    

In [27]:
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None
    

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [28]:
rag_retriever.retrieve("RENT ARREARS")

Retrieving documents for query: 'RENT ARREARS'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 60.24it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

In [41]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-20b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [43]:
answer=rag_simple("What is DEMAND FOR REFUND OF PROPERTY ADVANCE?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is DEMAND FOR REFUND OF PROPERTY ADVANCE?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 39.95it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


A formal legal notice that requests the return of an advance payment made toward a property transaction that did not go through. It demands repayment of the specified amount and seeks confirmation of a repayment date, warning that civil remedies may be pursued if the amount remains unpaid.


In [45]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("REFUND OF PROPERTY ADVANCE", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'REFUND OF PROPERTY ADVANCE'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 73.98it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: The notice demands that the INR 3,00,000 advance be refunded. It requests written confirmation of the repayment date and warns that civil remedies may be pursued if the amount remains unpaid.
Sources: [{'source': '04_property_payment_demand_notice_sample.pdf', 'page': 0, 'score': 0.11119675636291504, 'preview': 'SAMPLE / TEST DOCUMENT — NOT A REAL LEGAL NOTICE\n SAMPLE LEGAL NOTICE — DEMAND FOR REFUND OF PROPERTY\n ADVANCE\nNotice No.\nLL-TEST-2026-004\nDate\n14 August 2026\nTo\nMr. Sameer Desai\nAddress\n45, Residency Road, Bengaluru, Karnataka – 560025\nFrom\nMs. Nisha Patel\nAddress\n22, Richmond Town, Bengaluru, Karn...'}, {'source': '04_property_payment_demand_notice_sample.pdf', 'page': 0, 'score': 0.11119675636291504, 'preview': 'SAMPLE / TEST DOCUMENT — NOT A REAL LEGAL NOTICE\n SAMPLE LEGAL NOTICE — DEMAND FOR REFUND OF PROPERTY\n ADVANCE\nNotice No.\nLL-TEST-2026-004\nDate\n14 August 2026\nTo\nMr. Sameer Desai\nAddress\n45, Residency Road, Bengaluru, Karnataka – 56002

In [47]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what DEFECTIVE PRODUCT / CONSUMER DISPUTE", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what DEFECTIVE PRODUCT / CONSUMER DISPUTE'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 59.49it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
SAMPLE / TEST DOCUMENT — NOT A REAL LEGAL NOTICE
 SAMPLE LEGAL NOTICE — DEFECTIVE PRODUCT / CONSUMER
 DISPUTE
Notice No.
LL-TEST-2026-003
Date
14 August 2026
To
ABC Electronics Retail Pvt. Ltd.
Address
101, Commercial Complex, Bengaluru, Karnataka – 5

60001
From
Mr. Vivek Nair
Address
8, 2nd Cross, Bengaluru, Karnataka – 560040
Subject
Demand concerning allegedly defective electronic product
Purchase
The sender alleges that a fictional laptop was purchased from the recipient on 5 May 2026 for INR
78,500 under invoice number TEST-INV-260505.
Alleged Defect
The sender states that the laptop developed repeated display and charging failures during the alleged
warranty period despite two fictional service visits.
Relief Sought
The sender requests replacement of the product or refund of the alleged purchase price, together with
any other relief that may be legally available.
Response
The recipient is requested to respond within 15 days of receipt of the notice. This period is included for

SAMPLE / TEST DOCUMENT — NOT A REAL LEGAL NOTICE
 SAMPLE LEGAL NOTICE — DEFECTIVE PRODUCT / CONSUMER
 DISPUTE
Notice No.
LL-TEST-2026-003
Date
14 August 2026
To
ABC Electronics Retail Pvt. Ltd.
Address
101, Commercial Complex, Bengaluru, Karnataka – 560